# Breast Cancer (Wisconsin Original) 分类 + PDE 损失（PDE 预设解：量子真机一次性求解）

目标：分类任务总损失 = 分类损失 + λ·PDE 损失。PDE 的“预设解”只用量子真机（CIM）求一次；训练阶段的梯度与反向传播全部在经典端完成。

In [29]:
import os
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import kaiwu as kw

torch.manual_seed(7)
np.random.seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
kw.common.CheckpointManager.save_dir = "/tmp"

print("device:", device)


device: cuda


In [30]:
data_path = os.path.join("breast+cancer+wisconsin+original", "breast-cancer-wisconsin.data")

col_names = [
    "id",
    "clump_thickness",
    "uniformity_cell_size",
    "uniformity_cell_shape",
    "marginal_adhesion",
    "single_epithelial_cell_size",
    "bare_nuclei",
    "bland_chromatin",
    "normal_nucleoli",
    "mitoses",
    "class",
]

df = pd.read_csv(data_path, header=None, names=col_names)
df = df.replace("?", np.nan)

feature_cols = col_names[1:-1]
df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors="coerce")
df[feature_cols] = df[feature_cols].fillna(df[feature_cols].median(numeric_only=True))

y = (df["class"].astype(int).to_numpy() == 4).astype(np.float32)
X = df[feature_cols].to_numpy(dtype=np.float32)

X = (X - X.mean(axis=0, keepdims=True)) / (X.std(axis=0, keepdims=True) + 1e-6)

n = len(X)
perm = np.random.permutation(n)
split = int(n * 0.8)
train_idx, test_idx = perm[:split], perm[split:]

X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]

train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train).unsqueeze(1))
test_ds = TensorDataset(torch.tensor(X_test), torch.tensor(y_test).unsqueeze(1))

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

print("train:", X_train.shape, "test:", X_test.shape)


train: (559, 9) test: (140, 9)


## PDE：1D Poisson 方程（只做一次量子真机预设解）

取 1D 反应-扩散方程 $-\varepsilon u''(x) + u(x)=\sin(\pi x)$，$x\in(0,1)$，边界 $u(0)=u(1)=0$。当 $\varepsilon$ 较小，离散后的线性系统更对角占优，CIM 更容易在有限采样下找到更接近的解。

离散后得到线性系统 $A u=b$，用 QUBO 最小化 $\|A u-b\|^2$，再用 CIM 真机求一次“预设解”。

In [31]:
def build_reaction_diffusion_1d_system(
    num_interior: int, epsilon: float
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    h = 1.0 / (num_interior + 1)
    x = np.linspace(h, 1.0 - h, num_interior, dtype=float)
    main = (2.0 / (h * h)) * np.ones(num_interior, dtype=float)
    off = (-1.0 / (h * h)) * np.ones(num_interior - 1, dtype=float)
    lap = np.diag(main) + np.diag(off, k=1) + np.diag(off, k=-1)
    A = np.eye(num_interior, dtype=float) + float(epsilon) * lap
    b = np.sin(np.pi * x).astype(float)
    return x, A.astype(float), b.astype(float)


def build_qubo_for_least_squares(
    A: np.ndarray,
    b: np.ndarray,
    num_bits: int,
    value_scale: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    n = A.shape[0]
    weights = (2.0 ** np.arange(num_bits, dtype=float)).astype(float)
    offset = float(weights.sum() / 2.0)

    K = np.kron(np.eye(n, dtype=float), weights) * float(value_scale)
    u0 = (-offset * float(value_scale)) * np.ones(n, dtype=float)

    d = A @ u0 - b
    ATA = A.T @ A
    Q = K.T @ ATA @ K
    c = 2.0 * (K.T @ (A.T @ d.reshape(-1, 1))).reshape(-1)
    qubo = Q + np.diag(c)
    qubo = 0.5 * (qubo + qubo.T)
    qubo = kw.qubo.adjust_qubo_matrix_precision(qubo.astype(np.float32))
    return qubo, K.astype(np.float32), u0.astype(np.float32)


pde_n = 6
pde_epsilon = 1e-3
pde_x, pde_A, pde_b = build_reaction_diffusion_1d_system(pde_n, epsilon=pde_epsilon)
pde_u_classical = np.linalg.solve(pde_A, pde_b)

pde_num_bits = 6
pde_offset = float((2 ** pde_num_bits - 1) / 2)
value_scale = float(1.15 * np.max(np.abs(pde_u_classical)) / (pde_offset + 1e-12))
pde_qubo, pde_K, pde_u0 = build_qubo_for_least_squares(
    pde_A, pde_b, num_bits=pde_num_bits, value_scale=value_scale
)
n_qubo = int(pde_qubo.shape[0])

try:
    ising_mat, ising_bias = kw.conversion.qubo_matrix_to_ising_matrix(pde_qubo)
except Exception:
    ising_mat, ising_bias = kw.qubo.qubo_matrix_to_ising_matrix(pde_qubo)

n_vars = int(ising_mat.shape[0])
variables = [f"x[{i}]" for i in range(n_vars)]

pde_ising_model = None
try:
    pde_ising_model = kw.ising.IsingModel(variables=variables, ising_matrix=ising_mat, bias=ising_bias)
except Exception:
    pass

pde_submit_matrix = pde_ising_model.get_matrix() if pde_ising_model is not None else ising_mat

pde_task_name = f"pde_poisson1d_preset_{int(time.time())}"
pde_opt = kw.cim.CIMOptimizer(task_name=pde_task_name, task_mode="quota")
pde_opt.solve(pde_submit_matrix)

print("PDE preset 已提交到 CIM:", pde_task_name)
print("pde_n=", pde_n, "epsilon=", pde_epsilon, "qubo_vars=", n_qubo)
print("bits_per_u=", pde_num_bits, "value_scale=", value_scale)
print("classical u (first 3):", np.round(pde_u_classical[:3], 6))


[2026-05-24 15:59:46] [INFO    ] [kaiwu.cim._optimizer_adapter:68] - Task submit successfully, waiting for data validation. Task name: pde_poisson1d_preset_1779609585
PDE preset 已提交到 CIM: pde_poisson1d_preset_1779609585
pde_n= 6 epsilon= 0.001 qubo_vars= 36
bits_per_u= 6 value_scale= 0.03525049843458386
classical u (first 3): [0.429713 0.774317 0.965557]


In [32]:
sol = pde_opt.solve(pde_submit_matrix)
print("CIM 返回:", sol.shape)

if sol.shape[1] == n_qubo + 1:
    spins = sol[:, :n_qubo]
    deltas = sol[:, n_qubo]
    bits = (spins * deltas[:, np.newaxis] + 1.0) / 2.0
elif sol.shape[1] == n_qubo:
    spins = sol[:, :n_qubo]
    bits = (spins + 1.0) / 2.0
else:
    take = min(int(sol.shape[1]), n_qubo)
    spins = sol[:, :take]
    bits = (spins + 1.0) / 2.0
    if take != n_qubo:
        bits = np.pad(bits, ((0, 0), (0, n_qubo - take)), mode="constant")

bits = (bits > 0.5).astype(np.float64)

def bits_to_u(x: np.ndarray) -> np.ndarray:
    return (pde_u0.reshape(-1, 1) + pde_K @ x.reshape(-1, 1)).reshape(-1)

def residual_norm(u: np.ndarray) -> float:
    return float(np.linalg.norm(pde_A @ u - pde_b))

residuals = np.array([residual_norm(bits_to_u(bits[i])) for i in range(bits.shape[0])], dtype=float)
best_raw_idx = int(np.argmin(residuals))
best_raw_res = float(residuals[best_raw_idx])
best_raw_bits = bits[best_raw_idx]

print("best residual (raw):", best_raw_res)

def polish_bits_by_residual(x: np.ndarray, sweeps: int = 40) -> tuple[np.ndarray, float]:
    x = x.astype(np.float64, copy=True)
    n = int(x.shape[0])
    u = bits_to_u(x)
    best = residual_norm(u)
    for _ in range(int(sweeps)):
        improved = False
        for k in range(n):
            x[k] = 1.0 - x[k]
            r_new = residual_norm(bits_to_u(x))
            if r_new < best - 1e-9:
                best = r_new
                improved = True
            else:
                x[k] = 1.0 - x[k]
        if not improved:
            break
    return x, best

pde_q, best_residual = polish_bits_by_residual(best_raw_bits, sweeps=80)
pde_u_quantum = bits_to_u(pde_q)

rel_err = np.linalg.norm(pde_u_quantum - pde_u_classical) / (np.linalg.norm(pde_u_classical) + 1e-12)

pde_u_preset = torch.tensor(pde_u_quantum, dtype=torch.float32, device=device)

print("best residual (polished):", float(best_residual))
print("quantum preset u (first 3):", np.round(pde_u_quantum[:3], 6))
res_q = float(np.linalg.norm(pde_A @ pde_u_quantum - pde_b))
res_c = float(np.linalg.norm(pde_A @ pde_u_classical - pde_b))

print("relative error vs classical:", float(rel_err))
print("residual ||A u - b|| (quantum):", res_q)
print("residual ||A u - b|| (classical):", res_c)


[2026-05-24 15:59:54] [INFO    ] [kaiwu.cim._optimizer_adapter:1] - Task completed: pde_poisson1d_preset_1779609585
CIM 返回: (10, 37)
best residual (raw): 0.1593262749192541
best residual (polished): 0.04438360126390746
quantum preset u (first 3): [0.405381 0.757886 0.969389]
relative error vs classical: 0.022599959952896036
residual ||A u - b|| (quantum): 0.04438360126390746
residual ||A u - b|| (classical): 3.554447978966673e-16


## 分类网络 + PDE 损失

PDE 损失定义为 $\|u_{\theta}-u_{preset}\|^2$，其中 $u_{preset}$ 是上一步真机算出的预设解；$u_{\theta}$ 是网络里一个可训练的 PDE 参数向量。训练时只需要经典反向传播。

In [33]:
class ClassifierWithPDE(nn.Module):
    def __init__(self, in_dim: int, hidden: int, pde_size: int) -> None:
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )
        self.pde_field = nn.Parameter(torch.zeros(pde_size))
        self.pde_to_logit = nn.Parameter(torch.tensor(0.0))

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        logits = self.backbone(x)
        bias = self.pde_to_logit * self.pde_field.mean()
        return logits + bias, self.pde_field


model = ClassifierWithPDE(in_dim=X_train.shape[1], hidden=32, pde_size=pde_n).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
bce = nn.BCEWithLogitsLoss()

lambda_pde = 5.0

def evaluate(loader: DataLoader) -> tuple[float, float]:
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits, pde_field = model(xb)
            loss_cls = bce(logits, yb)
            loss_pde = torch.mean((pde_field - pde_u_preset) ** 2)
            loss = loss_cls + lambda_pde * loss_pde
            total_loss += float(loss) * xb.shape[0]
            pred = (torch.sigmoid(logits) > 0.5).to(torch.float32)
            correct += int((pred == yb).sum().item())
            total += int(xb.shape[0])
    return total_loss / max(1, total), correct / max(1, total)


for epoch in range(1, 31):
    model.train()
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        opt.zero_grad()
        logits, pde_field = model(xb)
        loss_cls = bce(logits, yb)
        loss_pde = torch.mean((pde_field - pde_u_preset) ** 2)
        loss = loss_cls + lambda_pde * loss_pde
        loss.backward()
        opt.step()

    if epoch % 5 == 0 or epoch == 1:
        tr_loss, tr_acc = evaluate(train_loader)
        te_loss, te_acc = evaluate(test_loader)
        u_err = float(torch.linalg.norm(model.pde_field.detach() - pde_u_preset) / (torch.linalg.norm(pde_u_preset) + 1e-12))
        print(
            f"epoch={epoch:02d} train_loss={tr_loss:.4f} train_acc={tr_acc:.3f} test_acc={te_acc:.3f} pde_rel_err={u_err:.3e}"
        )


epoch=01 train_loss=3.3826 train_acc=0.830 test_acc=0.871 pde_rel_err=9.886e-01
epoch=05 train_loss=2.8356 train_acc=0.966 test_acc=0.957 pde_rel_err=9.434e-01
epoch=10 train_loss=2.3157 train_acc=0.970 test_acc=0.957 pde_rel_err=8.888e-01
epoch=15 train_loss=2.0372 train_acc=0.973 test_acc=0.957 pde_rel_err=8.364e-01
epoch=20 train_loss=1.8023 train_acc=0.975 test_acc=0.957 pde_rel_err=7.863e-01
epoch=25 train_loss=1.5932 train_acc=0.975 test_acc=0.950 pde_rel_err=7.383e-01
epoch=30 train_loss=1.4064 train_acc=0.975 test_acc=0.950 pde_rel_err=6.925e-01
